# 06 — Gradual Virtual Lesion Analysis

Gradually disconnect ROIs in a target network (λ = 0.1 to 1.0) and compare against matched-size random null lesions outside that network.

In [1]:
import json
import multiprocessing as mp
import os
import time
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from tqdm import tqdm

PROJECT_ROOT = Path('..').resolve()
CONFIG_NAME = os.environ.get('GRAPH_THEORY_CONFIG', 'config.yaml')
with open(PROJECT_ROOT / CONFIG_NAME, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

paths = cfg['paths']
analysis = cfg['analysis']
graph_cfg = cfg['graph_theory']
RESULTS_DIR = (PROJECT_ROOT / paths['results_dir']).resolve()

K = int(analysis['K'])
method = analysis['method']
TARGET_NETWORK = graph_cfg['target_network']
graph_thresholds = np.asarray(graph_cfg['sparsity_thresholds'], dtype=float)
lesion_lambdas = np.asarray(graph_cfg['lesion_lambdas'], dtype=float)
n_null_replicates = int(graph_cfg['n_null_replicates'])
null_seed = int(graph_cfg['null_seed'])
use_parallel = bool(graph_cfg.get('use_parallel', True))
n_jobs = graph_cfg.get('n_jobs')

with open(RESULTS_DIR / 'analysis_metadata.json', encoding='utf-8') as f:
    meta = json.load(f)

select_networks = np.array(meta['select_networks'])

fc_npz = np.load(RESULTS_DIR / '04_weighted_fc' / 'weighted_average_FC.npz', allow_pickle=True)
weighted_avg_FC = fc_npz['weighted_avg_FC'].item()

# Load graph metric functions from notebook 03 when running standalone.
try:
    calculate_graph_metrics
except NameError:
    get_ipython().run_line_magic('run', '03_graph_metrics_functions.ipynb')

METRIC_KEYS = ['modularity', 'participation_coefficient']


consensus_runs=20, consensus_threshold=0.5


In [2]:
def apply_gradual_network_lesion(FC, net_indices, lesion_lambda, lesion_seed=42):
    FC_lesioned = FC.copy()
    lam = float(lesion_lambda)
    net_indices = np.asarray(net_indices, dtype=int)
    k_pool = len(net_indices)
    if lam <= 0 or k_pool == 0:
        return FC_lesioned, 0

    rng = np.random.RandomState(int(lesion_seed))
    ordered_rois = net_indices[rng.permutation(k_pool)]
    k_lesioned = k_pool if lam >= 1 else int(np.round(lam * k_pool))
    if k_lesioned <= 0:
        return FC_lesioned, 0

    lesion_rois = ordered_rois[:k_lesioned]
    R = FC.shape[0]
    FC_lesioned[np.ix_(lesion_rois, np.arange(R))] = 0
    FC_lesioned[np.ix_(np.arange(R), lesion_rois)] = 0
    return FC_lesioned, int(k_lesioned)


def _metric_from_result(metrics_dict, metric_key, prefix='global'):
    return metrics_dict.get(f'{prefix}_{metric_key}', np.nan)


def _trapz(y, x):
    fn = getattr(np, 'trapezoid', None) or np.trapz
    return fn(y, x)


def _compute_lesion_auc_summary(long_df, metric_cols, group_cols=('subject', 'lesion_lambda'), passthrough_cols=None):
    passthrough_cols = passthrough_cols or []
    auc_rows = []
    for group_key, group in long_df.groupby(list(group_cols)):
        row = dict(zip(group_cols, group_key if isinstance(group_key, tuple) else (group_key,)))
        for col in passthrough_cols:
            if col in group.columns:
                row[col] = group[col].iloc[0]
        thresholds = np.sort(group['graph_threshold'].unique())
        for col in metric_cols:
            values = group.sort_values('graph_threshold')[col].values
            if len(values) > 1 and np.isfinite(values).sum() > 1:
                valid_mask = np.isfinite(values)
                t = thresholds[valid_mask]
                v = values[valid_mask]
                row[f'{col}_auc'] = float(_trapz(v, t))
            elif len(values) == 1 and np.isfinite(values[0]):
                row[f'{col}_auc'] = float(values[0])
            else:
                row[f'{col}_auc'] = np.nan
        auc_rows.append(row)
    return pd.DataFrame(auc_rows)


In [3]:
def process_single_subject_gradual_lesion(subj_data, graph_thresholds, lesion_lambdas, sorted_networks, R, target_network):
    subj, FC = subj_data
    rows = []
    if FC is None or FC.shape[0] != R or np.all(np.isnan(FC)):
        return rows

    target_indices = np.where(sorted_networks == target_network)[0]
    if len(target_indices) < 3:
        return rows

    for graph_threshold in graph_thresholds:
        intact_metrics = calculate_graph_metrics(FC, graph_threshold, 'global', return_node_level=False)
        for lesion_lambda in lesion_lambdas:
            FC_lesioned, k_lesioned = apply_gradual_network_lesion(
                FC, target_indices, lesion_lambda, lesion_seed=42
            )
            lesion_metrics = calculate_graph_metrics(
                FC_lesioned, graph_threshold, 'lesion_global', return_node_level=False
            )
            row = {
                'subject': subj,
                'lesion_lambda': float(lesion_lambda),
                'graph_threshold': float(graph_threshold),
                'target_network': target_network,
                'k_roi_pool': int(len(target_indices)),
                'k_lesioned': int(k_lesioned),
            }
            for key in METRIC_KEYS:
                intact_val = _metric_from_result(intact_metrics, key)
                lesion_val = _metric_from_result(lesion_metrics, key, prefix='lesion_global')
                delta_val = intact_val - lesion_val if np.isfinite(intact_val) and np.isfinite(lesion_val) else np.nan
                if key == 'modularity':
                    row['modularity_intact'] = intact_val
                    row['modularity_lesion'] = lesion_val
                    row['delta_modularity'] = delta_val
                else:
                    row[f'{key}_intact'] = intact_val
                    row[f'{key}_lesion'] = lesion_val
                    row[f'delta_{key}'] = delta_val
            rows.append(row)
    return rows


def run_gradual_network_lesion_analysis(
    weighted_avg_FC,
    sorted_networks,
    target_network,
    graph_thresholds,
    lesion_lambdas,
    use_parallel=True,
    n_jobs=None,
    save_dir='results/gradual_lesion',
):
    valid_subjects = [s for s, fc in weighted_avg_FC.items() if fc is not None]
    R = weighted_avg_FC[valid_subjects[0]].shape[0]
    subj_data_list = [(s, weighted_avg_FC[s]) for s in valid_subjects]
    process_func = partial(
        process_single_subject_gradual_lesion,
        graph_thresholds=graph_thresholds,
        lesion_lambdas=lesion_lambdas,
        sorted_networks=sorted_networks,
        R=R,
        target_network=target_network,
    )
    if n_jobs is None:
        n_jobs = min(mp.cpu_count(), len(valid_subjects))

    all_rows = []
    if use_parallel:
        with mp.Pool(processes=n_jobs) as pool:
            for batch in tqdm(pool.imap(process_func, subj_data_list), total=len(subj_data_list), desc='Gradual lesion'):
                all_rows.extend(batch)
    else:
        for item in tqdm(subj_data_list, desc='Gradual lesion'):
            all_rows.extend(process_func(item))

    os.makedirs(save_dir, exist_ok=True)
    long_df = pd.DataFrame(all_rows)
    tag = f'1000parcels_{K}states_{method}'
    long_path = Path(save_dir) / f'gradual_lesion_long_{tag}.csv'
    long_df.to_csv(long_path, index=False)

    metric_cols = ['modularity_intact', 'modularity_lesion', 'delta_modularity']
    for key in METRIC_KEYS:
        if key != 'modularity':
            metric_cols.extend([f'{key}_intact', f'{key}_lesion', f'delta_{key}'])

    auc_df = _compute_lesion_auc_summary(
        long_df, metric_cols, passthrough_cols=['k_lesioned', 'k_roi_pool']
    )
    auc_path = Path(save_dir) / f'gradual_lesion_auc_{tag}.csv'
    auc_df.to_csv(auc_path, index=False)
    print(f'Saved {long_path.name} and {auc_path.name}.')
    return long_df, auc_df


In [4]:
def generate_random_null_roi_sets(sorted_networks, n_null, seed, target_network):
    target_indices = np.where(sorted_networks == target_network)[0]
    non_target_pool = np.where(sorted_networks != target_network)[0]
    k = len(target_indices)
    if k < 3:
        raise ValueError(f'Target network {target_network} has fewer than 3 ROIs.')
    if k > len(non_target_pool):
        raise ValueError('Non-target ROI pool smaller than target network size.')

    rng = np.random.RandomState(seed)
    null_sets = []
    manifest_rows = []
    for null_id in range(1, int(n_null) + 1):
        chosen = np.sort(rng.choice(non_target_pool, size=k, replace=False))
        null_sets.append(chosen)
        for roi_idx in chosen:
            manifest_rows.append({
                'null_id': null_id,
                'roi_index': int(roi_idx),
                'network': sorted_networks[roi_idx],
                'k_matched_target': int(k),
                'target_network': target_network,
            })
    return null_sets, pd.DataFrame(manifest_rows), int(k)


def _build_intact_lookup(long_df):
    if long_df is None or len(long_df) == 0:
        return {}
    intact_cols = ['modularity_intact']
    for key in METRIC_KEYS:
        if key != 'modularity':
            intact_cols.append(f'{key}_intact')
    lookup = {}
    base = long_df.groupby(['subject', 'graph_threshold'], as_index=False)[intact_cols].first()
    for _, row in base.iterrows():
        lookup[(str(row['subject']), float(row['graph_threshold']))] = row.to_dict()
    return lookup


def _append_gradual_lesion_rows(rows, subj, FC, lesion_indices, graph_thresholds, lesion_lambdas, intact_lookup, lesion_prefix, extra_meta):
    for gt in graph_thresholds:
        intact_row = intact_lookup.get((str(subj), float(gt)))
        if intact_row is not None:
            intact_metrics = {
                key: intact_row.get('modularity_intact' if key == 'modularity' else f'{key}_intact', np.nan)
                for key in METRIC_KEYS
            }
        else:
            intact_metrics_raw = calculate_graph_metrics(FC, gt, 'global', return_node_level=False)
            intact_metrics = {key: _metric_from_result(intact_metrics_raw, key) for key in METRIC_KEYS}

        for lesion_lambda in lesion_lambdas:
            FC_lesioned, k_lesioned = apply_gradual_network_lesion(
                FC, lesion_indices, lesion_lambda, lesion_seed=extra_meta.get('lesion_seed', 42)
            )
            lesion_metrics = calculate_graph_metrics(FC_lesioned, gt, lesion_prefix, return_node_level=False)
            row = {
                'subject': subj,
                'lesion_lambda': float(lesion_lambda),
                'graph_threshold': float(gt),
                'k_roi_pool': int(len(lesion_indices)),
                'k_lesioned': int(k_lesioned),
            }
            row.update({k: v for k, v in extra_meta.items() if k != 'k_lesioned'})
            for key in METRIC_KEYS:
                intact_val = intact_metrics[key]
                lesion_val = _metric_from_result(lesion_metrics, key, prefix=lesion_prefix)
                delta_val = intact_val - lesion_val if np.isfinite(intact_val) and np.isfinite(lesion_val) else np.nan
                if key == 'modularity':
                    row['modularity_intact'] = intact_val
                    row['modularity_lesion_null'] = lesion_val
                    row['delta_modularity_null'] = delta_val
                else:
                    row[f'{key}_intact'] = intact_val
                    row[f'{key}_lesion_null'] = lesion_val
                    row[f'delta_{key}_null'] = delta_val
            rows.append(row)


def process_single_null_replicate(null_task):
    null_id, null_indices, subj_data_list, graph_thresholds, lesion_lambdas, R, intact_lookup = null_task
    rows = []
    extra_meta = {
        'null_id': int(null_id),
        'lesion_type': 'random_non_target',
        'lesion_seed': int(null_id) * 10007 + 42,
    }
    for subj, FC in subj_data_list:
        if FC is None or FC.shape[0] != R or np.all(np.isnan(FC)):
            continue
        _append_gradual_lesion_rows(
            rows, subj, FC, null_indices, graph_thresholds, lesion_lambdas,
            intact_lookup, 'lesion_null_global', extra_meta,
        )
    return rows


def run_random_null_lesion_analysis(
    weighted_avg_FC,
    sorted_networks,
    target_network,
    graph_thresholds,
    lesion_lambdas,
    n_null_replicates,
    null_seed,
    intact_reference_df=None,
    use_parallel=True,
    n_jobs=None,
    save_dir='results/random_null_lesion',
):
    null_sets, manifest_df, k_target = generate_random_null_roi_sets(
        sorted_networks, n_null_replicates, null_seed, target_network
    )
    valid_subjects = [s for s, fc in weighted_avg_FC.items() if fc is not None]
    R = weighted_avg_FC[valid_subjects[0]].shape[0]
    subj_data_list = [(s, weighted_avg_FC[s]) for s in valid_subjects]
    intact_lookup = _build_intact_lookup(intact_reference_df)

    null_tasks = [
        (null_id, null_indices, subj_data_list, graph_thresholds, lesion_lambdas, R, intact_lookup)
        for null_id, null_indices in enumerate(null_sets, start=1)
    ]
    if n_jobs is None:
        n_jobs = min(mp.cpu_count(), len(null_tasks))

    all_rows = []
    if use_parallel:
        with mp.Pool(processes=n_jobs) as pool:
            for batch in tqdm(pool.imap(process_single_null_replicate, null_tasks), total=len(null_tasks), desc='Null replicates'):
                all_rows.extend(batch)
    else:
        for task in null_tasks:
            all_rows.extend(process_single_null_replicate(task))

    os.makedirs(save_dir, exist_ok=True)
    tag = f'1000parcels_{K}states_{method}'
    manifest_path = Path(save_dir) / f'random_null_roi_sets_{tag}.csv'
    manifest_df.to_csv(manifest_path, index=False)

    long_df = pd.DataFrame(all_rows)
    long_path = Path(save_dir) / f'random_null_lesion_long_{tag}.csv'
    long_df.to_csv(long_path, index=False)

    metric_cols = ['modularity_intact', 'modularity_lesion_null', 'delta_modularity_null']
    for key in METRIC_KEYS:
        if key != 'modularity':
            metric_cols.extend([f'{key}_intact', f'{key}_lesion_null', f'delta_{key}_null'])

    auc_df = _compute_lesion_auc_summary(
        long_df, metric_cols, group_cols=('subject', 'lesion_lambda', 'null_id'), passthrough_cols=['k_lesioned', 'k_roi_pool']
    )
    auc_path = Path(save_dir) / f'random_null_lesion_auc_{tag}.csv'
    auc_df.to_csv(auc_path, index=False)
    print(f'Saved null outputs to {save_dir}.')
    return long_df, auc_df, manifest_df


In [5]:
gradual_dir = RESULTS_DIR / 'gradual_lesion'
gradual_long_df, gradual_auc_df = run_gradual_network_lesion_analysis(
    weighted_avg_FC=weighted_avg_FC,
    sorted_networks=select_networks,
    target_network=TARGET_NETWORK,
    graph_thresholds=graph_thresholds,
    lesion_lambdas=lesion_lambdas,
    use_parallel=use_parallel,
    n_jobs=n_jobs,
    save_dir=str(gradual_dir),
)

null_dir = RESULTS_DIR / 'random_null_lesion'
null_long_df, null_auc_df, null_manifest_df = run_random_null_lesion_analysis(
    weighted_avg_FC=weighted_avg_FC,
    sorted_networks=select_networks,
    target_network=TARGET_NETWORK,
    graph_thresholds=graph_thresholds,
    lesion_lambdas=lesion_lambdas,
    n_null_replicates=n_null_replicates,
    null_seed=null_seed,
    intact_reference_df=gradual_long_df,
    use_parallel=use_parallel,
    n_jobs=n_jobs,
    save_dir=str(null_dir),
)

print('Gradual lesion and random null analyses complete.')
print(f' target network: {TARGET_NETWORK}')
print(f' matched k: {null_manifest_df["k_matched_target"].iloc[0]}')
print(f' gradual rows: {len(gradual_long_df)}, null rows: {len(null_long_df)}')


Gradual lesion:   0%|          | 0/3 [00:00<?, ?it/s]

Gradual lesion:  33%|███▎      | 1/3 [00:00<00:01,  1.03it/s]

Gradual lesion:  67%|██████▋   | 2/3 [00:01<00:00,  1.01it/s]

Gradual lesion: 100%|██████████| 3/3 [00:02<00:00,  1.03it/s]

Gradual lesion: 100%|██████████| 3/3 [00:02<00:00,  1.02it/s]

Saved gradual_lesion_long_1000parcels_3states_example.csv and gradual_lesion_auc_1000parcels_3states_example.csv.


Saved null outputs to D:\文件同步\文件\数据分析结果\data_code_figure\adaptive_reserve_moderate\graph_theory\results\example\random_null_lesion.
Gradual lesion and random null analyses complete.
 target network: SalVentAttn
 matched k: 6
 gradual rows: 90, null rows: 450
